# BirdCLEF 2026 - Submission v40 - Strong TTA

Same model as v34/v38 (`exp78_full_model.keras`, val AUC 0.8533, prior LB 0.592).

**What is new vs v34_3view_tta:**
1. **5-view temporal TTA**: offsets [-1.0, -0.5, 0.0, 0.5, 1.0] (was 3)
2. **Geometric mean aggregation** (changes ranking; ROC-AUC is rank-based)
3. **2 light SpecAugment TTA passes** averaged with the temporal views
4. **Removed sqrt calibration** (monotone per-row -> no effect on ROC-AUC)
5. **Per-class temperature scaling-free** (also rank-invariant, removed)


In [ ]:
import os, glob, json, shutil, zipfile, warnings
import numpy as np
import pandas as pd
import librosa
import tensorflow as tf
from tensorflow import keras

warnings.filterwarnings('ignore')

print('TensorFlow:', tf.__version__)
print('Librosa:', librosa.__version__)

In [ ]:
# =======================================================
# PATHS + EXP78 PARAMETERS
# =======================================================

ORIGINAL_MODEL = '/kaggle/input/datasets/danielemalerba0302/birdclef2026-model/exp78_full_model.keras'
TEST_AUDIO_DIR = '/kaggle/input/competitions/birdclef-2026/test_soundscapes'
SAMPLE_SUB_PATH = '/kaggle/input/competitions/birdclef-2026/sample_submission.csv'
SUBMISSION_PATH = '/kaggle/working/submission.csv'

SAMPLE_RATE = 32000
DURATION = 5.0

N_MELS = 64
N_FFT = 2048
HOP_LENGTH = 256
FMIN = 20
FMAX = 16000
TOP_DB = 40.0
MEL_NORM = 'slaney'
USE_HTK = True

TARGET_HEIGHT = 64
TARGET_WIDTH = 626

# =======================================================
# TTA CONFIG  - tune to balance time-budget (90 min CPU)
# Each row_id costs (len(TTA_OFFSETS) + N_SPECAUG) forward passes.
# Default = 5 + 2 = 7 passes per row.
# =======================================================
TTA_OFFSETS = [-1.0, -0.5, 0.0, 0.5, 1.0]
N_SPECAUG_TTA = 2
USE_GEOMETRIC_MEAN = True
BATCH_SIZE = 32

print('Test dir exists:', os.path.exists(TEST_AUDIO_DIR))
print('Sample submission exists:', os.path.exists(SAMPLE_SUB_PATH))

assert os.path.exists(ORIGINAL_MODEL), f'Missing model: {ORIGINAL_MODEL}'
assert os.path.exists(SAMPLE_SUB_PATH), f'Missing sample submission: {SAMPLE_SUB_PATH}'

In [ ]:
# Sample submission / species order
sample_sub = pd.read_csv(SAMPLE_SUB_PATH)
SPECIES_LIST = list(sample_sub.columns[1:])
print('Sample submission shape:', sample_sub.shape)
print('Number of species:', len(SPECIES_LIST))
assert len(SPECIES_LIST) == 234

In [ ]:
# =======================================================
# PATCH .keras CONFIG FOR KAGGLE KERAS COMPATIBILITY
# =======================================================
PATCHED_MODEL = '/kaggle/working/exp78_full_model_patched.keras'
tmp_dir = '/kaggle/working/keras_patch_tmp'

if os.path.exists(tmp_dir):
    shutil.rmtree(tmp_dir)
os.makedirs(tmp_dir)

with zipfile.ZipFile(ORIGINAL_MODEL, 'r') as z:
    z.extractall(tmp_dir)

config_path = os.path.join(tmp_dir, 'config.json')
with open(config_path, 'r') as f:
    config = json.load(f)

BAD_KEYS = ['renorm', 'renorm_clipping', 'renorm_momentum', 'quantization_config']

def remove_bad_keys(obj):
    if isinstance(obj, dict):
        for key in BAD_KEYS:
            obj.pop(key, None)
        for value in obj.values():
            remove_bad_keys(value)
    elif isinstance(obj, list):
        for item in obj:
            remove_bad_keys(item)

remove_bad_keys(config)

with open(config_path, 'w') as f:
    json.dump(config, f)

with zipfile.ZipFile(PATCHED_MODEL, 'w', zipfile.ZIP_DEFLATED) as z:
    for root, _, files in os.walk(tmp_dir):
        for file in files:
            full_path = os.path.join(root, file)
            arcname = os.path.relpath(full_path, tmp_dir)
            z.write(full_path, arcname)

print('Patched model saved:', PATCHED_MODEL)

In [ ]:
model = keras.models.load_model(PATCHED_MODEL, compile=False, safe_mode=False)
print('Input shape:', model.input_shape)
print('Output shape:', model.output_shape)
assert model.input_shape[1:] == (TARGET_HEIGHT, TARGET_WIDTH, 3)
assert model.output_shape[-1] == len(SPECIES_LIST)

In [ ]:
# =======================================================
# PREPROCESSING - MUST MATCH EXP78 TRAINING PIPELINE
# =======================================================

def audio_to_spectrogram(audio_arr, sr=SAMPLE_RATE):
    mel = librosa.feature.melspectrogram(
        y=audio_arr, sr=sr,
        n_mels=N_MELS, n_fft=N_FFT, hop_length=HOP_LENGTH,
        fmin=FMIN, fmax=FMAX,
        norm=MEL_NORM, htk=USE_HTK
    )
    mel = np.nan_to_num(mel, nan=0.0, posinf=0.0, neginf=0.0)
    mel_db = librosa.power_to_db(mel, ref=np.max, top_db=TOP_DB)
    mel_db = np.nan_to_num(mel_db, nan=-TOP_DB, posinf=0.0, neginf=-TOP_DB)
    mel_norm = (mel_db + TOP_DB) / TOP_DB
    mel_norm = np.clip(mel_norm, 0, 1)
    spec = np.stack([mel_norm, mel_norm, mel_norm], axis=-1).astype(np.float32)
    if spec.shape[1] < TARGET_WIDTH:
        spec = np.pad(spec, ((0, 0), (0, TARGET_WIDTH - spec.shape[1]), (0, 0)), mode='constant')
    elif spec.shape[1] > TARGET_WIDTH:
        spec = spec[:, :TARGET_WIDTH, :]
    return spec

def light_specaugment(spec, rng):
    """Apply 1 small freq-mask + 1 small time-mask. Used for TTA.
    Light parameters because we average with un-augmented views."""
    spec = spec.copy()
    H, W, C = spec.shape
    fw = rng.randint(2, 8)
    fs = rng.randint(0, max(1, H - fw))
    spec[fs:fs+fw, :, :] = 0.0
    tw = rng.randint(5, 25)
    ts = rng.randint(0, max(1, W - tw))
    spec[:, ts:ts+tw, :] = 0.0
    return spec

# Sanity check
dummy = np.zeros(int(SAMPLE_RATE * DURATION), dtype=np.float32)
test_spec = audio_to_spectrogram(dummy)
print('Test spec shape:', test_spec.shape, ' Expected:', model.input_shape[1:])
assert test_spec.shape == model.input_shape[1:]

In [ ]:
# =======================================================
# TTA EXTRACTION
# =======================================================

def extract_shifted_block(y, start_second, offset_second):
    block_samples = int(DURATION * SAMPLE_RATE)
    start_sample = int(round((start_second + offset_second) * SAMPLE_RATE))
    end_sample = start_sample + block_samples
    left_pad = max(0, -start_sample)
    right_pad = max(0, end_sample - len(y))
    start_sample = max(0, start_sample)
    end_sample = min(len(y), end_sample)
    block = y[start_sample:end_sample]
    if left_pad or right_pad:
        block = np.pad(block, (left_pad, right_pad), mode='constant')
    if len(block) < block_samples:
        block = np.pad(block, (0, block_samples - len(block)), mode='constant')
    elif len(block) > block_samples:
        block = block[:block_samples]
    return block

def build_tta_specs_for_row(y, end_second, rng):
    """Build all TTA specs for a single row_id.
    Returns list of (variant_name, spec). Total = N_OFFSETS + N_SPECAUG_TTA.
    """
    start_second = end_second - int(DURATION)
    out = []
    # Temporal TTA
    for off in TTA_OFFSETS:
        block = extract_shifted_block(y, start_second, off)
        spec = audio_to_spectrogram(block, sr=SAMPLE_RATE)
        out.append((f'shift_{off:+.2f}', spec))
    # SpecAugment TTA over the centered window only
    centered_block = extract_shifted_block(y, start_second, 0.0)
    centered_spec = audio_to_spectrogram(centered_block, sr=SAMPLE_RATE)
    for k in range(N_SPECAUG_TTA):
        out.append((f'specaug_{k}', light_specaugment(centered_spec, rng)))
    return out

In [ ]:
# =======================================================
# AGGREGATION HELPER
# arithmetic OR geometric mean over TTA predictions for same row.
# Geometric mean of probs is more conservative on outliers.
# =======================================================
_LOG_EPS = 1e-7

def aggregate_tta(pred_list):
    arr = np.stack(pred_list, axis=0)  # (T, n_classes)
    if USE_GEOMETRIC_MEAN:
        log_arr = np.log(np.clip(arr, _LOG_EPS, 1.0))
        agg = np.exp(log_arr.mean(axis=0))
    else:
        agg = arr.mean(axis=0)
    return np.clip(agg, 0.0, 1.0)

In [ ]:
# =======================================================
# INFERENCE LOOP
# =======================================================

audio_files = []
for ext in ['*.ogg', '*.wav', '*.flac', '*.mp3']:
    audio_files.extend(glob.glob(os.path.join(TEST_AUDIO_DIR, '**', ext), recursive=True))
audio_files = sorted(audio_files)

print('Found test audio files:', len(audio_files))
print('TTA offsets:', TTA_OFFSETS, 'SpecAug TTA:', N_SPECAUG_TTA, 'Geom mean:', USE_GEOMETRIC_MEAN)

expected_by_file = {}
for row_id in sample_sub['row_id']:
    file_stem = '_'.join(row_id.split('_')[:-1])
    expected_by_file.setdefault(file_stem, []).append(row_id)

predictions_by_row_id = {}
batch_specs = []
batch_row_ids = []

def flush_batch():
    global batch_specs, batch_row_ids
    if not batch_specs:
        return
    X = np.array(batch_specs, dtype=np.float32)
    preds = model.predict(X, verbose=0)
    preds = np.clip(preds, 0.0, 1.0)
    for row_id, pred in zip(batch_row_ids, preds):
        predictions_by_row_id.setdefault(row_id, []).append(pred)
    batch_specs = []
    batch_row_ids = []

rng = np.random.RandomState(20260507)

for i, audio_path in enumerate(audio_files):
    file_stem = os.path.splitext(os.path.basename(audio_path))[0]
    if i % 5 == 0:
        print(f'{i+1}/{len(audio_files)}: {file_stem}')
    if file_stem not in expected_by_file:
        continue
    y, _ = librosa.load(audio_path, sr=SAMPLE_RATE, mono=True)
    for row_id in expected_by_file[file_stem]:
        end_sec = int(row_id.split('_')[-1])
        for _, spec in build_tta_specs_for_row(y, end_sec, rng):
            batch_row_ids.append(row_id)
            batch_specs.append(spec)
            if len(batch_specs) >= BATCH_SIZE:
                flush_batch()

flush_batch()
print('Rows with predictions:', len(predictions_by_row_id))

In [ ]:
# =======================================================
# AGGREGATE TTA + BUILD SUBMISSION
# =======================================================

expected_row_ids = list(sample_sub['row_id'])

if len(predictions_by_row_id) == 0:
    print('No predictions generated -> placeholder zeros (visible-run fallback)')
    arr = np.zeros((len(expected_row_ids), len(SPECIES_LIST)), dtype=np.float32)
    submission_df = pd.DataFrame(arr, columns=SPECIES_LIST)
    submission_df.insert(0, 'row_id', expected_row_ids)
else:
    missing = sorted(set(expected_row_ids) - set(predictions_by_row_id.keys()))
    extra = sorted(set(predictions_by_row_id.keys()) - set(expected_row_ids))
    print('Missing rows:', len(missing), '  Extra rows:', len(extra))
    assert not extra, f'Unexpected row_ids: {extra[:5]}'
    rows = []
    for rid in expected_row_ids:
        if rid in predictions_by_row_id:
            rows.append(aggregate_tta(predictions_by_row_id[rid]))
        else:
            rows.append(np.zeros(len(SPECIES_LIST), dtype=np.float32))
    arr = np.stack(rows, axis=0).astype(np.float32)
    submission_df = pd.DataFrame(arr, columns=SPECIES_LIST)
    submission_df.insert(0, 'row_id', expected_row_ids)

submission_df.to_csv(SUBMISSION_PATH, index=False)
print('Saved:', SUBMISSION_PATH, '  shape:', submission_df.shape)
submission_df.head()

In [ ]:
# Verification
check_df = pd.read_csv(SUBMISSION_PATH)
sample_check = pd.read_csv(SAMPLE_SUB_PATH)
print('Rows:', len(check_df), 'Cols:', len(check_df.columns))
assert list(check_df.columns) == list(sample_check.columns)
assert list(check_df['row_id']) == list(sample_check['row_id'])
vals = check_df.iloc[:, 1:].values
print('min:', vals.min(), 'max:', vals.max(), 'NaN:', np.isnan(vals).any())
assert not np.isnan(vals).any()
assert vals.min() >= 0 and vals.max() <= 1
print('Submission ready.')